# 070 — Bahdanau against Luong Attention

Two ways to compute the alignment score. The difference matters because one of
them becomes self-attention: Bahdanau scores with a **learned network**
($2H^2 + H$ parameters), Luong with a **dot product** (**zero**). At $H = 1000$
that is **2,001,000 against none**.

| Part | What we check |
|---|---|
| A | the two score functions, computed on the same inputs |
| B | the parameter counts |
| C | the second difference: which decoder state, and where the context goes |
| D | scaling, and why the dot product needs it |

## Part A — Two scores, same inputs

In [ ]:
import numpy as np

rng = np.random.default_rng(0)
T, H = 5, 16

def softmax(z, axis=-1):
    z = z - z.max(axis, keepdims=True)
    e = np.exp(z)
    return e / e.sum(axis, keepdims=True)

encoder_states = rng.normal(0, 1, (T, H))
decoder_state = rng.normal(0, 1, H)

# Bahdanau: additive. A one-layer network over the concatenation.
W1 = rng.normal(0, 0.3, (H, H))      # on the decoder state
W2 = rng.normal(0, 0.3, (H, H))      # on each encoder state
v = rng.normal(0, 0.3, H)            # the output projection
additive = np.array([v @ np.tanh(W1 @ decoder_state + W2 @ h)
                     for h in encoder_states])

# Luong: multiplicative. Three variants, two of which have no network at all.
dot = encoder_states @ decoder_state
Wg = rng.normal(0, 0.3, (H, H))
general = encoder_states @ (Wg @ decoder_state)

for name, scores in (("Bahdanau additive", additive), ("Luong dot", dot),
                     ("Luong general", general)):
    print(f"  {name:<18} {scores.round(2)}  -> peak {scores.argmax()}")
    assert abs(softmax(scores).sum() - 1) < 1e-9
print()
print("All three produce one number per encoder state, and the softmax over")
print("them is the attention weight. The difference is entirely in HOW the")
print("number is produced.")

## Part B — What each one costs

In [ ]:
print(f"  {'H':>6}{'additive':>14}{'general':>12}{'dot':>8}")
for h in (256, 512, 1000):
    print(f"  {h:>6}{2 * h * h + h:>14,}{h * h:>12,}{0:>8}")

assert 2 * 1000 ** 2 + 1000 == 2_001_000
assert 1000 ** 2 == 1_000_000
print()
print("At H = 1000: 2,001,000 parameters for the additive network, 1,000,000")
print("for the general form, and none at all for the plain dot product.")
print()
print("The dot product works only because both vectors already live in the same")
print("space and the same width. The additive form does not need that - it")
print("learns a projection for each side, which is why it survives when the")
print("encoder and decoder widths differ.")
assert encoder_states.shape[1] == decoder_state.shape[0]

## Part C — The second difference

Bahdanau scores against the **previous** decoder state $s_{i-1}$ and feeds the
context in as **input**. Luong scores against the **current** state $s_i$ and
concatenates the context on the **output** side. That changes the order of
operations inside one step.

In [ ]:
Wo = rng.normal(0, 0.3, (2 * H, H))

# Bahdanau: context first, then the decoder step consumes it
context_b = softmax(additive) @ encoder_states
bahdanau_input = np.concatenate([context_b, decoder_state])    # fed IN
print("Bahdanau: score(s_{i-1}, h) -> context -> decoder step")
print(f"   the decoder's input is {bahdanau_input.shape[0]} wide "
      f"(context {H} + embedding {H})")

# Luong: decoder step first, then the context is combined with its output
state_now = np.tanh(decoder_state @ rng.normal(0, 0.3, (H, H)))  # s_i
context_l = softmax(encoder_states @ state_now) @ encoder_states
luong_output = np.tanh(np.concatenate([context_l, state_now]) @ Wo)  # combined OUT
print("Luong:    decoder step -> score(s_i, h) -> context -> combine")
print(f"   the combined output is {luong_output.shape[0]} wide")

assert bahdanau_input.shape == (2 * H,)
assert luong_output.shape == (H,)
print()
print("Same ingredients, different order. Luong's version is the one that reads")
print("like modern attention - query, keys, values, then a projection - which")
print("is why the dot-product form is the one the transformer kept.")

## Part D — Why the dot product needs scaling

A dot product of two $H$-dimensional vectors grows with $H$, and a softmax of
large numbers is a one-hot vector with no gradient.

In [ ]:
print(f"  {'H':>6}{'mean |score|':>14}{'max softmax':>13}{'scaled max':>12}")
rows = []
for h in (8, 64, 512, 4096):
    g = np.random.default_rng(0)
    q = g.normal(0, 1, h)
    keys = g.normal(0, 1, (T, h))
    raw = keys @ q
    scaled = raw / np.sqrt(h)
    rows.append((h, abs(raw).mean(), softmax(raw).max(), softmax(scaled).max()))
    print(f"  {h:>6}{abs(raw).mean():>14.1f}{softmax(raw).max():>13.3f}"
          f"{softmax(scaled).max():>12.3f}")

assert rows[-1][1] > rows[0][1] * 5           # the scores grow with H
assert rows[-1][2] > 0.99                     # unscaled softmax saturates
assert rows[-1][3] < rows[-1][2]              # scaling pulls it back
print()
print("At H = 4096 the unscaled softmax puts essentially all its weight on one")
print("position, so the gradient through the other positions vanishes. Dividing")
print("by sqrt(H) keeps the scores in a usable range - which is exactly the")
print("scaled dot-product attention of the transformer (lesson 075), and the")
print("reason the scaling factor is there at all.")

## What this notebook established

- Bahdanau (additive) and Luong (multiplicative) both produce one score per
  encoder state; only the computation differs.
- Parameters at $H = 1000$: **2,001,000** additive, **1,000,000** for Luong's
  general form, **0** for the plain dot product — which only works because both
  vectors share a space and a width.
- The second difference is structural: Bahdanau scores against $s_{i-1}$ and
  feeds the context in, Luong scores against $s_i$ and concatenates it on the
  way out.
- An unscaled dot product saturates the softmax as $H$ grows — at $H = 4096$ one
  position took over **99%** of the weight — which is why scaled dot-product
  attention divides by $\sqrt{H}$.

## Exercises

1. Make the encoder and decoder widths differ. Which score functions still
   work, and what has to be added to the others?
2. Plot the maximum softmax weight against $H$ for scaled and unscaled scores.
   Where do the lines separate?
3. Count the parameters of a full Bahdanau layer for $H = 512$ including the
   output projection, and compare it with one transformer attention head.